# AIOps Train Autoencoder

Purpose: train PyTorch autoencoders on clean baseline feature tables. 

In [0]:
%pip install torch --quiet

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import json, os
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

STORAGE_ACCOUNT = "hantstorageaccount"
LAKEHOUSE_CONTAINER = "lakehouse"

dbutils.widgets.text('model_version', 'v1')
MODEL_VERSION = dbutils.widgets.get('model_version')
BASE = f'abfss://{LAKEHOUSE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net'
FEATURE_ROOT = f'{BASE}/monitoring/aiops/features'
MODEL_STORAGE_ROOT = f'{BASE}/monitoring/aiops/model_registry/{MODEL_VERSION}'
LOCAL_MODEL_ROOT = f'/dbfs/tmp/aiops_model_registry/{MODEL_VERSION}'
DBFS_MODEL_ROOT = f'dbfs:/tmp/aiops_model_registry/{MODEL_VERSION}'
os.makedirs(LOCAL_MODEL_ROOT, exist_ok=True)
torch.manual_seed(42)
np.random.seed(42)

/local_disk0/.ephemeral_nfs/envs/pythonEnv-b620e228-54e5-4f5a-89d5-1c7eaa33881b/lib/python3.12/site-packages/torch/_vmap_internals.py:9: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  from torch.utils._pytree import _broadcast_to_and_flatten, tree_flatten, tree_unflatten


In [0]:
SILVER_HEADER_FEATURES = [
    'invoice_id_present',
    'order_date_present',
    'customer_name_present',
    'ship_mode_present',
    'source_type_code',
    'ship_mode_code',
    'BalanceDue',
    'SubTotal',
    'DiscountPercent',
    'DiscountAmount',
    'ShippingAmount',
    'InvoiceTotal',
    'line_count',
    'sum_line_subtotal',
    'avg_quantity',
    'avg_unit_price',
    'calc_invoice_total',
    'header_total_diff',
    'header_subtotal_rollup_diff',
    'discount_ratio',
    'missing_header_field_count',
    'missing_financial_field_count',
    'abs_header_total_diff',
    'abs_header_subtotal_rollup_diff',
    'shipping_ratio',
    'balance_due_ratio',
    'line_count_log',
    'avg_line_amount',
    'max_line_subtotal',
    'stddev_line_subtotal',
]
SILVER_LINES_FEATURES = [
    'invoice_id_present',
    'line_number_present',
    'product_name_present',
    'product_id_present',
    'quantity_present',
    'unit_price_present',
    'item_subtotal_present',
    'source_type_code',
    'product_name_code',
    'Quantity',
    'UnitPrice',
    'ItemSubTotal',
    'calc_item_subtotal',
    'line_subtotal_diff',
    'abs_line_subtotal_diff',
    'line_amount_ratio_to_invoice',
    'quantity_log',
    'unit_price_log',
    'item_subtotal_log',
]

TRAINING_JOBS = [
    ('silver_header_autoencoder',
     f'{FEATURE_ROOT}/training/header/', SILVER_HEADER_FEATURES, 80),
    ('silver_lines_autoencoder',
     f'{FEATURE_ROOT}/training/lines/', SILVER_LINES_FEATURES, 80)
]


In [0]:
class Autoencoder(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        latent_dim = max(2, min(8, input_dim // 2)) # Use half the input size but never smaller than 2 and larger than 8
        hidden_dim = max(8, input_dim * 2)
        self.net = nn.Sequential(
            # Encoder: Compress input into a smaller latent representation
            nn.Linear(input_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, latent_dim), nn.ReLU(),
            # Decoder: Reconstruct the original input from the latent representation
            nn.Linear(latent_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, input_dim))
    def forward(self, x):
        return self.net(x)

def load_training_matrix(path, feature_cols):
    df = spark.read.format('delta').load(path)
    pdf = df.select(*feature_cols).fillna(0.0).toPandas().astype('float32')
    if len(pdf) < 20:
        raise RuntimeError(f'Need at least 20 rows for training, found {len(pdf)} at {path}')
    x = pdf.values.astype('float32') # convert to numpy array
    mean = x.mean(axis=0).astype('float32') # calculate the mean of each feature
    std = x.std(axis=0).astype('float32') # calculate the standard deviation of each feature
    std[std == 0] = 1.0
    return ((x - mean) / std).astype('float32'), mean, std # return normalized data, mean, and standard deviation

In [0]:
def train_model(model_name, feature_path, feature_cols, epochs):
    x, mean, std = load_training_matrix(feature_path, feature_cols)
    tensor = torch.tensor(x, dtype=torch.float32) # convert numpy matrix to PyTorch tensor
    # Create batches for training
    loader = DataLoader(
        TensorDataset(tensor), # wrap the tensor into a PyTorch Dataset
        batch_size=min(64, len(tensor)), # use batch size 64 if there are at least 64 rows, otherwise use the number of rows
        shuffle=True # randomizes row order each epoch
    )
    # Create an autoencoder
    model = Autoencoder(len(feature_cols))
    # Create an optimizer with learning rate 0.001, update the model weights during training
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    # Create a loss function, measure the difference between the original data and the reconstructed data (MSE = Mean Squared Error)
    loss_fn = nn.MSELoss()
    for epoch in range(epochs):
        for (batch,) in loader:
            optimizer.zero_grad() # clear old gradients from the previous training step
            loss = loss_fn(model(batch), batch) # Run the batch through the autoencoder and calculate the loss
            loss.backward() # calculate the gradients based on the loss
            optimizer.step() # update the model weights to reduce future reconstruction error
    model.eval() # set the model to evaluation mode
    # Disable gradient tracking and calculate the reconstruction error for each row
    with torch.no_grad():
        errors = torch.mean((model(tensor) - tensor) ** 2, dim=1).numpy()
    threshold = float(np.quantile(errors, 0.975)) # If error is above 97.5% of the training data, the row is considered anomalous
    model_dir = os.path.join(LOCAL_MODEL_ROOT, model_name)
    os.makedirs(model_dir, exist_ok=True)
    torch.save(model.state_dict(), os.path.join(model_dir, 'model.pt')) # save the model weights into model.pt
    metadata = {
        'model_name': model_name,
        'model_version': MODEL_VERSION,
        'input_dim': len(feature_cols),
        'feature_columns': feature_cols,
        'mean': mean.tolist(),
        'std': std.tolist(),
        'threshold': threshold,
        'training_rows': int(len(x))
    }
    with open(os.path.join(model_dir, 'metadata.json'), 'w', encoding='utf-8') as f:
        json.dump(metadata, f, indent=2)
    print(f'{model_name}: rows={len(x)}, threshold={threshold:.6f}')
    return metadata

In [0]:
trained = []
for model_name, feature_path, feature_cols, epochs in TRAINING_JOBS:
    try:
        trained.append(train_model(model_name, feature_path, feature_cols, epochs))
    except Exception as e:
        print(f'Skipped {model_name}: {e}')

with open(os.path.join(LOCAL_MODEL_ROOT, 'training_summary.json'), 'w', encoding='utf-8') as f:
    json.dump(trained, f, indent=2)

dbutils.fs.cp(DBFS_MODEL_ROOT, MODEL_STORAGE_ROOT, True)
dbutils.jobs.taskValues.set(key='aiops_model_version', value=MODEL_VERSION)
dbutils.jobs.taskValues.set(key='aiops_model_storage_root', value=MODEL_STORAGE_ROOT)
print(f'Published AIOps models to {MODEL_STORAGE_ROOT}')

silver_header_autoencoder: rows=2300, threshold=0.152823
silver_lines_autoencoder: rows=6879, threshold=0.000835
Published AIOps models to abfss://lakehouse@hantstorageaccount.dfs.core.windows.net/monitoring/aiops/model_registry/v1
